# Corpus load and inspection

The 20Newsgroup corpus consists of a series of emails. To extract meaningful text, is necessary to focus on the body of the emails, leaving out headers and metadata, previously cited emails, signatures, among other data.

In [2]:
# Libraries and setup

# File handling
from pathlib import Path
import os
import random
import re

# Data handling
import pandas as pd
import numpy as np

# NLP
import spacy

# Display settings
pd.set_option("display.max_colwidth", 1000)

# Reproducibility

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [3]:
# Load NLP model

nlp = spacy.load("en_core_web_sm")

In [4]:
# Corpus path
# I am using the mini_newsgroup corpus uploaded to drive. Replace path as necessary.

corpus_path = Path("/content/drive/MyDrive/UoW_PETER/mini_newsgroups")

In [5]:
# Collect all files

all_files = list(corpus_path.rglob("*"))

# Keep only files
all_files = [f for f in all_files if f.is_file()]

print("Total files:", len(all_files))

Total files: 2000


It is important to manually review a number of files to have an idea of the overall format and decide correctly the preprocessing steps.

In [6]:
# Inspect random files
# Re-run as many times as needed

sample_files = random.sample(
    all_files,
    min(5, len(all_files))
)

for file_path in sample_files:

    print("=" * 80)
    print("FILE:", file_path.name)
    print("LABEL:", file_path.parent.name)
    print("=" * 80)

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print(text[:3000])
    print("\n\n")

FILE: 61087
LABEL: sci.space
Newsgroups: sci.space
Path: cantaloupe.srv.cs.cmu.edu!rochester!udel!gatech!howland.reston.ans.net!zaphod.mps.ohio-state.edu!sdd.hp.com!decwrl!apple!mumbo.apple.com!gallant.apple.com!kip-37.apple.com!user
From: keithley@apple.com (Craig Keithley)
Subject: Re: Moonbase race, NASA resources, why?
Sender: news@gallant.apple.com
Message-ID: <keithley-220493104229@kip-37.apple.com>
Date: Thu, 22 Apr 1993 18:36:42 GMT
References: <C5sx3y.3z9.1@cs.cmu.edu> <C5tEIK.7z9@zoo.toronto.edu> <1r46o9INN14j@mojo.eng.umd.edu> <1993Apr21.210712.1@aurora.alaska.edu> <C5w5un.Bpq@zoo.toronto.edu>
Organization: Apple Computer, Inc.
Followup-To: sci.space
Lines: 44

In article <C5w5un.Bpq@zoo.toronto.edu>, henry@zoo.toronto.edu (Henry
Spencer) wrote:
> 
> The major component of any realistic plan to go to the Moon cheaply (for
> more than a brief visit, at least) is low-cost transport to Earth orbit.
> For what it costs to launch one Shuttle or two Titan IVs, you can develop
> a 

**Corpus observations and preprocessing strategy**


After reviewing random corpus samples, the following patterns were identified:

1. Metadata leakage. Risk of data leakage
* The "Xref" line contains the target label.
* Metadata/header sections must therefore be removed.

2. Email body extraction
* Email content generally starts after " writes:"
* Some files contain multiple occurrences.
* Some files do not contain the pattern.

3. Quoted email chains
* Previous replies often start with ">"
* These should likely be removed.

4. Line breaks
* Emails contain multiple line jumps.
* Line breaks do not necessarily indicate different sentences.

5. Email signatures
* Signatures are often very short lines.
* Usually 1–2 words long.
* These may be removable during cleaning (alongside everything that comes after)

# Prototype corpus cleaning

In [7]:
# Function 1
# Remove metadata/header section

#def remove_metadata(text):

    # Split after "Lines: <integer>"
   # parts = re.split(
    #    r"Lines:\s*\d+",
    #    text,
    #    flags=re.IGNORECASE
  #  )

    # If metadata separator exists
  #  if len(parts) > 1:
  #      return parts[-1]

    # Fallback
  #  return text

Remaining observed noise patterns:

* Metadata tags (indicates "Lines:" is not the last metadata tag for every email)

In next implementation, all known metadata tags will be removed explicitly.

In [8]:
# Function 1
# Remove metadata/header lines

def remove_metadata(text):

    metadata_patterns = [
        r"^Newsgroups:",
        r"^From:",
        r"^Subject:",
        r"^Organization:",
        r"^Lines:",
        r"^Distribution:",
        r"^NNTP-Posting-Host:",
        r"^Message-ID:",
        r"^References:",
        r"^Path:",
        r"^Reply-To:",
        r"^Date:",
        r"^Keywords:",
        r"^Summary:",
        r"^Expires:",
        r"^Sender:",
        r"^Article-I\.D\.:",
        r"^Nntp-Posting-Host:",
        r"^Xref:"
    ]

    cleaned_lines = []

    for line in text.splitlines():

        line_stripped = line.strip()

        # Skip metadata lines
        if any(
            re.match(pattern, line_stripped, flags=re.IGNORECASE)
            for pattern in metadata_patterns
        ):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

In [9]:
# Function 2
# Remove quoted email lines

def remove_quoted_lines(text):

    cleaned_lines = []

    for line in text.splitlines():

        # Remove quoted replies
        if line.strip().startswith(">"):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

In [10]:
# Function 3
# Extract email body

def extract_email_body(text):

    # Find all occurrences of "writes:"
    matches = list(
        re.finditer(
            r"writes\s*:",
            text,
            flags=re.IGNORECASE
        )
    )

    # Use the last occurrence
    if matches:

        last_match = matches[-1]

        return text[last_match.end():]

    return text

In [11]:
# Function 4
# Reconstruct broken lines

def reconstruct_text(text):

    lines = []

    for line in text.splitlines():

        # Remove empty lines
        line = line.strip()

        if not line:
            continue

        lines.append(line)

    # Rebuild paragraph
    text = " ".join(lines)

    return text

In [12]:
# Function 5
# Final text cleanup

def clean_whitespace(text):

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [13]:
# Prototype cleaning pipeline

def clean_email(text):

    # Remove metadata/header
    text = remove_metadata(text)

    # Remove quoted replies
    text = remove_quoted_lines(text)

    # Extract main body
    text = extract_email_body(text)

    # Reconstruct paragraphs
    text = reconstruct_text(text)

    # Final cleanup
    text = clean_whitespace(text)

    return text

# Inspect prototype cleaning

In [16]:
sample_files = random.sample(
    all_files,
    min(5, len(all_files))
)

for file_path in sample_files:

    print("=" * 80)
    print("FILE:", file_path.name)
    print("=" * 80)

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

    cleaned_text = clean_email(raw_text)

    print("\nRAW TEXT:\n")
    print(raw_text[:1500])

    print("\nCLEANED TEXT:\n")
    print(cleaned_text[:1500])

    print("\n\n")

FILE: 61066

RAW TEXT:

Newsgroups: sci.space
Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!bb3.andrew.cmu.edu!crabapple.srv.cs.cmu.edu!nickh
From: nickh@CS.CMU.EDU (Nick Haines)
Subject: Re: Vandalizing the sky.
In-Reply-To: todd@phad.la.locus.com's message of Wed, 21 Apr 93 16:28:00 GMT
Message-ID: <C5w5F8.3LC.1@cs.cmu.edu>
Originator: nickh@SNOW.FOX.CS.CMU.EDU
Sender: news@cs.cmu.edu (Usenet News System)
Nntp-Posting-Host: snow.fox.cs.cmu.edu
Organization: School of Computer Science, Carnegie Mellon University
References: <C5t05K.DB6@research.canon.oz.au>
	<1993Apr21.162800.168967@locus.com>
Date: Thu, 22 Apr 1993 15:23:17 GMT
Lines: 33

In article <1993Apr21.162800.168967@locus.com> todd@phad.la.locus.com (Todd Johnson) writes:

   As for advertising -- sure, why not?  A NASA friend and I spent one
   drunken night figuring out just exactly how much gold mylar we'd need
   to put the golden arches of a certain American fast food organization
   on the f

# Building clean document corpus

In [17]:
# Build cleaned document corpus

records = []

for file_path in all_files:

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        raw_text = f.read()

    cleaned_text = clean_email(raw_text)

    records.append(
        {
            "textid": file_path.stem,
            "label": file_path.parent.name,
            "raw_text": raw_text,
            "cleaned_text": cleaned_text
        }
    )

df_docs = pd.DataFrame(records)

print(f"Documents: {len(df_docs):,}")
display(df_docs.head())

Documents: 2,000


,textid,label,raw_text,cleaned_text
0,76062,misc.forsale,"Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!howland.reston.ans.net!noc.near.net!nic.umass.edu!titan.ucs.umass.edu!not-for-mail\nFrom: klj@titan.ucs.umass.edu (KATHERINE L JEFFERS)\nNewsgroups: misc.forsale\nSubject: MAC SE FORSALE\nDate: 20 Apr 1993 13:18:56 -0400\nOrganization: University of Massachusetts, Amherst\nLines: 11\nDistribution: usa\nMessage-ID: <1r1ba0INN7gg@titan.ucs.umass.edu>\nNNTP-Posting-Host: titan.ucs.umass.edu\n\nThis is a repost of an earlier. Thanks to several of you for\noffering advise on realistic prices.\n\nMAC SE/ 2.5 megs ram, 20 meg hard disk, 800 K Floppy.\nIn absolutely perfect condition. \n\nIncludes Word 5, pagemaker, quark xpress, quicken and the \nlatest versions of about a dozen other programs.\n\nPrice: 475.00\n\n","This is a repost of an earlier. Thanks to several of you for offering advise on realistic prices. MAC SE/ 2.5 megs ram, 20 meg hard disk, 800 K Floppy. In absolutely perfect condition. Includes Word 5, pagemaker, quark xpress, quicken and the latest versions of about a dozen other programs. Price: 475.00"
1,70337,misc.forsale,"Path: cantaloupe.srv.cs.cmu.edu!rochester!udel!gatech!howland.reston.ans.net!usc!cs.utexas.edu!qt.cs.utexas.edu!news.Brown.EDU!noc.near.net!bigboote.WPI.EDU!bigwpi.WPI.EDU!kedz\nFrom: kedz@bigwpi.WPI.EDU (John Kedziora)\nNewsgroups: misc.forsale\nSubject: Motorcycle wanted.\nDate: 22 Feb 1993 14:22:51 GMT\nOrganization: Worcester Polytechnic Institute\nLines: 11\nExpires: 5/1/93\nMessage-ID: <1manjr$ja0@bigboote.WPI.EDU>\nNNTP-Posting-Host: bigwpi.wpi.edu\n\nSender: \nFollowup-To:kedz@wpi.wpi.edu \nDistribution: ne\nOrganization: Worcester Polytechnic Institute\nKeywords: \n\nI am looking for an inexpensive motorcycle, nothing fancy, have to be able to do all maintinence my self. looking in the <$400 range.\n\nif you can help me out, GREAT!, please reply by e-mail.\n\n\n","Followup-To:kedz@wpi.wpi.edu I am looking for an inexpensive motorcycle, nothing fancy, have to be able to do all maintinence my self. looking in the <$400 range. if you can help me out, GREAT!, please reply by e-mail."
2,74731,misc.forsale,"Path: cantaloupe.srv.cs.cmu.edu!das-news.harvard.edu!ogicse!network.ucsd.edu!usc!zaphod.mps.ohio-state.edu!saimiri.primate.wisc.edu!doug.cae.wisc.edu!beng\nFrom: beng@cae.wisc.edu (Beng Ting)\nNewsgroups: misc.forsale,wi.forsale,uwisc.forsale\nSubject: Madison/Chicago --> Italy Air Ticket Wanted\nKeywords: Madison/Chicago, Milan, Italy\nMessage-ID: <1993Apr5.193913.14385@doug.cae.wisc.edu>\nDate: 6 Apr 93 00:39:13 GMT\nArticle-I.D.: doug.1993Apr5.193913.14385\nOrganization: U of Wisconsin-Madison College of Engineering\nLines: 13\n\nHi,\n\n\tI am looking for a round trip Madison/Chicago --> Milan (Italy)\n\tair ticket. Anybody who has a transferable ticket but will\n\tnot use it please contact me at beng@cae.wisc.edu. Open-jaw\n\tticket highly desired.\n\t\n\tThank you.\n\n\nB. T. Ting\nbeng@cae.wisc.edu\n \n","Hi, I am looking for a round trip Madison/Chicago --> Milan (Italy) air ticket. Anybody who has a transferable ticket but will not use it please contact me at beng@cae.wisc.edu. Open-jaw ticket highly desired. Thank you. B. T. Ting beng@cae.wisc.edu"
3,76499,misc.forsale,"Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!bb3.andrew.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!howland.reston.ans.net!gatech!emory!ogicse!clark!pro-freedom.van.wa.us!bbates\nFrom: bbates@pro-freedom.van.wa.us (Brandon Bates)\nNewsgroups: misc.forsale\nSubject: WANTED: Video equipment (repost)\nMessage-ID: <1993Apr21.155915.10932@pro-freedom.van.wa.us>\nDate: 21 Apr 93 22:59:15 GMT\nArticle-I.D.: pro-free.1993Apr21.155915.10932\nSender: usenet@clark.edu\nOrganization: ProLine [pro-freedom] AppleVan (Apple UG of Vancouver, WA)\nLines: 11\n\n\n I am looking for a working docking deck (deck that goes on b

In [18]:
df_docs["textid"].nunique()
len(df_docs)

2000

The previous block shows that each text id is unique.

In [20]:
# Save cleaned document corpus

output_file = "20Newsgroup_Mini_cleaned_document_corpus.csv"

df_docs.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Saved: {output_file}")

Saved: 20Newsgroup_Mini_cleaned_document_corpus.csv


**Validations**

In [21]:
empty_docs = (
    df_docs["cleaned_text"]
    .str.strip()
    .eq("")
    .sum()
)

print(f"Empty documents: {empty_docs}")

Empty documents: 3


In [22]:
doc_lengths = df_docs["cleaned_text"].str.len()

print(doc_lengths.describe())

count     2000.000000
mean      1283.138500
std       3650.615425
min          0.000000
25%        353.250000
50%        630.000000
75%       1163.500000
max      67001.000000
Name: cleaned_text, dtype: float64


In [24]:
# Shortest documents

df_docs.loc[
    df_docs["cleaned_text"].str.len().nsmallest(10).index,
    ["textid", "label", "cleaned_text"]
]

,textid,label,cleaned_text
1209,101675,rec.autos,
1805,76401,talk.politics.mideast,
1865,77250,talk.politics.mideast,
1436,51522,comp.sys.mac.hardware,art
330,10790,comp.os.ms-windows.misc,Perhaps. ;-)
470,104414,rec.sport.baseball,"Japan, I think."
1784,54748,talk.politics.guns,Dave 1 Clinton 0
289,53603,alt.atheism,"In a word, yes. mathew"
1183,53786,rec.sport.hockey,Need I say more???????
1982,83785,talk.religion.misc,"No, J.R. ""Bob"" Dobbs. mathew"


In [40]:
display(
    df_docs.sample(
        10,
        #random_state=RANDOM_STATE
    )[
        ["textid", "label", "cleaned_text"]
    ]
)

,textid,label,cleaned_text
889,53918,sci.electronics,"I have the ARRL Handbook for the Radio AMateur, and I'm getting the Solid STate Design for the Radio Amateur. The handbook is $25 and Solid State is $12 from ARRL, 225 Main, Newington, CT 06111 but you should be able to find them at electronics or amateur radio stores. ARRL will want $5 or so for shipping. Good Luck, Galen Watts, KF0YJ"
1065,16029,sci.crypt,"Deeply grateful for citations to any papers on electronic cash schemes. Enquiring minds &c... -- Eric Weaver Sony AVTC 677 River Oaks Pkwy, MS 35 SJ CA 95134 408 944-4904 & Chief Engineer, KFJC 89.7 Foothill College, Los Altos Hills CA 94022"
761,59284,sci.med,"------------- cut here ----------------- HICNet Medical Newsletter Page 13 Volume 6, Number 11 April 25, 1993 :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::: Food & Drug Administration News :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::: FDA Approves Depo Provera, injectable contraceptive P92-31 Food and Drug Administration FOR IMMEDIATE RELEASE Susan Cruzan - (301) 443-3285 The Food and Drug Administration today announced the approval of Depo Provera, an injectable contraceptive drug. The drug, which contains a synthetic hormone similar to the natural hormone progesterone, protects women from pregnancy for three months per injection. The hormone is injected into the muscle of the arm or buttock where it is released into the bloodstream to prevent pregnancy. It is more than 99 percent effective. ""This drug presents another long-term, effective option for women to prevent pregnancy,"" said FDA Commissioner David A. Kessler..."
560,38266,comp.graphics,"I am looking for EISA or VESA local bus graphic cards that support at least 1024x786x24 resolution. I know Matrox has one, but it is very expensive. All the other cards I know of, that support that resoultion, are striaght ISA. Also are there any X servers for a unix PC that support 24 bits? thanks"
427,102606,rec.sport.baseball,Neither did he! Overall? How do you figure? So far my radio hasn't exploded from not being tuned to 660... Roger
1513,20886,soc.religion.christian,"If you agree that good works have a role somewhere, you will generally find yourself in one of two camps: (1) Faith + Works --> Salvation or (2) Faith --> Salvation + Works Either (1) works are required for salvation, or (2) faith will inevitably result in good works. I am also of the opinion that salvation is by faith alone, based on Ephesians 2 and Romans 3:21-31. I also conclude that James 2, when read in context, is teaching bullet (2) above. When James speaks of justification, I would claim that he is not speaking of God declaring the believing sinner innocent in His sight (Paul's use of the word). Instead he is speaking of the sinner's profession of faith being ""justified"" or ""proven"" by the display of good works. Also according to James 2, the abscence of such works is evidence for a ""dead"" or ""useless"" faith which fails to save. James 2 is not a problem for the doctrine of salvation by faith if it is teaching (2). Works would have their place, not as merit toward salvation,..."
1166,54136,rec.sport.hockey,"|> |> As a person who has rarely even SEEN Don Cherry and doesn't know |> anything about him, I don't know whether it is just this area |> (Pittsburgh) of the USA that is ""deprived"" of his broadcasts or whether |> he's a Canadian thing altogether. Seriously, what is he all about? I |> know he was a coach at one time, and from the volume of posts about him, |> SOMEONE surely is getting a steady diet of him somehow, but my question |> is, what is the deal with him? Secondly, are the comments of his that I |> read about on the net merely flame bait, or do people actually take him |> seriously? I gotta tell you, from what I see, he really sounds like an |> ass. Let me know - maybe I'm missing something. I asked him, and Don Cherry denies being Roger Maynard and he denie